In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
    cross_validate,
    GridSearchCV
)


from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


from sklearn.decomposition import PCA
from lightgbm import LGBMClassifier

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

In [ ]:
train_data = pd.read_csv(path + '/fashion-mnist_train.csv')
test_data = pd.read_csv(path + '/fashion-mnist_test.csv')

In [ ]:
train_data.shape, test_data.shape

In [ ]:
class_labels = ['T-shirt/top',
                'Trouser',
                'Pullover',
                'Dress',
                'Coat',
                'Sandal',
                'Shirt',
                'Sneaker',
                'Bag',
                'Ankle boot']

In [ ]:
plt.figure(figsize=(12, 4))
for i in range(20):
  plt.subplot(2, 10, i + 1)
  plt.imshow(train_data.values[i][1:].reshape(28, 28), cmap='binary', interpolation='nearest')
  plt.xlabel(class_labels[train_data.values[i][0]])

plt.tight_layout()

In [ ]:
X_train = train_data.drop('label', axis=1)
X_test = test_data.drop('label', axis=1)

y_train = train_data['label']
y_test = test_data['label']

In [ ]:
X_train.shape, y_train.shape

In [ ]:
np.unique(y_train)

In [ ]:
y_train.value_counts()

In [ ]:
X_train, X_test = X_train/255., X_test/255.

pca = PCA(n_components=0.85)

model = LGBMClassifier(n_jobs = -1)

X_train_reduced = pca.fit_transform(X_train)
X_test_reduced = pca.transform(X_test)



sss = StratifiedShuffleSplit(n_splits=4, test_size=0.2, random_state=42)


models_list = []
precision_scores = []
recall_scores = []
f1_scores = []


# 4 fold cross-validation
for i, (train_indices, test_indices) in enumerate(sss.split(X_train_reduced, y_train)):
  X_train_sss, y_train_sss = X_train_reduced[train_indices], y_train[train_indices]
  X_test_sss, y_test_sss = X_train_reduced[test_indices], y_train[test_indices]

  model.fit(X_train_sss, y_train_sss)
  predictions = model.predict(X_test_sss)

  prec_ = precision_score(y_test_sss, predictions, average='weighted')
  rec_ = recall_score(y_test_sss, predictions, average='weighted')
  f1_ = f1_score(y_test_sss, predictions, average='weighted')

  print('precision_score : {}'.format(prec_))
  print('recall_score : {}'.format(rec_))
  print('f1_score : {}'.format(f1_))

  precision_scores.append(prec_)
  recall_scores.append(rec_)
  f1_scores.append(f1_)

print('\n'*3)

results = pd.DataFrame({'precisions': precision_scores, 'recalls': recall_scores, 'f1 scores': f1_scores})
print(results)


print('\n'*2)
print('mean results for cross validations: ')
print(results.mean(axis=0))

In [ ]:
model.get_params()

In [ ]:
param_grid = {
  'n_estimators': [100, 150],
  'max_depth': [-1, 20],
  'min_child_samples': [10, 20, 40],
  'learning_rate': [0.1, 0.5],

}


X_train_reduced_sample, _, y_train_sample, _ = train_test_split(X_train_reduced, y_train, train_size=0.1, random_state=42, stratify=y_train)



grid_search_cv = GridSearchCV(model, param_grid, cv=3, n_jobs=-1, verbose=4)
grid_search_cv.fit(X_train_reduced_sample, y_train_sample)


print(f'best score found with grid search (training - test): {grid_search_cv.best_score_}')
print()
print(f'best parameters: {grid_search_cv.best_params_}')


print('\n'*3)


print(pd.DataFrame(grid_search_cv.cv_results_))

print('best estimator score on test data (projected) from GridSearchCV'.\
      format(f1_score(y_test, grid_search_cv.best_estimator_.predict(X_test_reduced), average='weighted')))

In [ ]:
grid_search_cv.best_estimator_.score(X_test_reduced, y_test)

In [ ]:
predictions = grid_search_cv.best_estimator_.predict(X_test_reduced)

print('f1 score : {}'.format(f1_score(y_test, predictions, average='weighted')))
print('precision score : {}'.format(precision_score(y_test, predictions, average='weighted')))
print('recall score : {}'.format(recall_score(y_test, predictions, average='weighted')))